In [6]:
import numpy as np
import torch as th
import json

keep_pair_sj= [2, 56, 130, 308, 348, 367, 510, 737, 805, 824, 1024, 1066, 1098, 1149, 1263, 1432, 1482, 1498, 1503, 1605]
print(len(keep_pair_sj))

file = '/home/mint/Dev/DiFaReli/difareli-faster/experiment_scripts/TPAMI/sample_json/TR/gen_pair_TR_for_rotate.json'
with open(file, 'r') as f:
    data = json.load(f)
    print(len(data['pair']))

out = {'pair':{}}
for i in keep_pair_sj:
    out['pair'][f'pair{i}'] = {'src':data['pair'][f'pair{i}']['src'],
                               'dst':'60065.jpg',
    }

print(len(out['pair']))
with open('/home/mint/Dev/DiFaReli/difareli-faster/experiment_scripts/TPAMI/sample_json/DiFaReli++/HDRI_sota_sj.json', 'w') as f:
    json.dump(out, f, indent=4)

20
1902
20


# Process for final folder structure

In [ ]:
from PIL import Image
import glob, os, tqdm
import numpy as np

def face_segment(segment_part, img):
    
    if isinstance(img, Image.Image):
        face_segment_anno = np.array(img)
    else:
        face_segment_anno = img
        
    bg = (face_segment_anno == 0)
    skin = (face_segment_anno == 1)
    l_brow = (face_segment_anno == 2)
    r_brow = (face_segment_anno == 3)
    l_eye = (face_segment_anno == 4)
    r_eye = (face_segment_anno == 5)
    eye_g = (face_segment_anno == 6)
    l_ear = (face_segment_anno == 7)
    r_ear = (face_segment_anno == 8)
    ear_r = (face_segment_anno == 9)
    nose = (face_segment_anno == 10)
    mouth = (face_segment_anno == 11)
    u_lip = (face_segment_anno == 12)
    l_lip = (face_segment_anno == 13)
    neck = (face_segment_anno == 14)
    neck_l = (face_segment_anno == 15)
    cloth = (face_segment_anno == 16)
    hair = (face_segment_anno == 17)
    hat = (face_segment_anno == 18)
    face = np.logical_or.reduce((skin, l_brow, r_brow, l_eye, r_eye, eye_g, l_ear, r_ear, ear_r, nose, mouth, u_lip, l_lip))
    foreground = face_segment_anno != 0

    if segment_part == 'faceseg_face':
        seg_m = face
    elif segment_part == 'faceseg_foreground':
        seg_m = foreground
    elif segment_part == 'faceseg_head':
        seg_m = (face | neck | hair)
    elif segment_part == 'faceseg_nohead':
        seg_m = ~(face | neck | hair)
    elif segment_part == 'faceseg_face&hair':
        seg_m = ~bg
    elif segment_part == 'faceseg_bg_noface&nohair':
        seg_m = (bg | hat | neck | neck_l | cloth) 
    elif segment_part == 'faceseg_bg&ears_noface&nohair':
        seg_m = (bg | hat | neck | neck_l | cloth) | (l_ear | r_ear | ear_r)
    elif segment_part == 'faceseg_bg':
        seg_m = bg
    elif segment_part == 'faceseg_bg&noface':
        seg_m = (bg | hair | hat | neck | neck_l | cloth)
    elif segment_part == 'faceseg_hair':
        seg_m = hair
    elif segment_part == 'faceseg_faceskin':
        seg_m = skin
    elif segment_part == 'faceseg_faceskin&nose':
        seg_m = (skin | nose)
    elif segment_part == 'faceseg_eyes&glasses&mouth&eyebrows':
        seg_m = (l_eye | r_eye | eye_g | l_brow | r_brow | mouth)
    elif segment_part == 'faceseg_faceskin&nose&mouth&eyebrows':
        seg_m = (skin | nose | mouth | u_lip | l_lip | l_brow | r_brow | l_eye | r_eye)
    elif segment_part == 'faceseg_faceskin&nose&mouth&eyebrows&eyes&glasses':
        seg_m = (skin | nose | mouth | u_lip | l_lip | l_brow | r_brow | l_eye | r_eye | eye_g)
    elif segment_part == 'faceseg_face_noglasses':
        seg_m = (~eye_g & face)
    elif segment_part == 'faceseg_face_noglasses_noeyes':
        seg_m = (~(l_eye | r_eye) & ~eye_g & face)
    elif segment_part == 'faceseg_eyes&glasses':
        seg_m = (l_eye | r_eye | eye_g)
    elif segment_part == 'glasses':
        seg_m = eye_g
    elif segment_part == 'faceseg_eyes':
        seg_m = (l_eye | r_eye)
    # elif (segment_part == 'sobel_bg_mask') or (segment_part == 'laplacian_bg_mask') or (segment_part == 'sobel_bin_bg_mask'):
    elif segment_part in ['sobel_bg_mask', 'laplacian_bg_mask', 'sobel_bin_bg_mask']:
        seg_m = ~(face | neck | hair)
    elif segment_part in ['canny_edge_bg_mask']:
        seg_m = ~(face | neck | hair) | (l_ear | r_ear)
    else: raise NotImplementedError(f"Segment part: {segment_part} is not found!")
    
    out = seg_m
    return out

data_path = '/data/mint/DPM_Dataset/HDRI_Dataset/Testset/subject_512x512/'
out_path = '/data/mint/DPM_Dataset/HDRI_Dataset/Testset/mask_512x512/'
os.makedirs(out_path, exist_ok=True)

for f in tqdm.tqdm(glob.glob(data_path + '/*.png')):
    src_id = f.split('/')[-1]
    faceseg_dir = f"/data/mint/DPM_Dataset/HDRI_Dataset/Testset/face_segment_512x512/anno/anno_{src_id.split('.')[0]}.png"
    faceseg = np.array(Image.open(faceseg_dir))
    foreground_m = face_segment('faceseg_foreground', faceseg)
    Image.fromarray(foreground_m.astype(np.uint8)*255).save(f"{out_path}/{src_id.split('.')[0]}.png")
    
data_path = '/data/mint/DPM_Dataset/HDRI_Dataset/Testset/subject_256x256/'
out_path = '/data/mint/DPM_Dataset/HDRI_Dataset/Testset/mask_256x256/'
os.makedirs(out_path, exist_ok=True)

for f in tqdm.tqdm(glob.glob(data_path + '/*.jpg')):
    src_id = f.split('/')[-1]
    faceseg_dir = f"/data/mint/DPM_Dataset/ffhq_256_with_anno/face_segment_with_pupil/valid/anno/anno_{src_id.split('.')[0]}.png"
    faceseg = np.array(Image.open(faceseg_dir))
    foreground_m = face_segment('faceseg_foreground', faceseg)
    Image.fromarray(foreground_m.astype(np.uint8)*255).save(f"{out_path}/{src_id.split('.')[0]}.png")


100%|██████████| 20/20 [00:00<00:00, 32.80it/s]


## Create dataset folder
- 2 folders
1. subjects
2. env that contains map named with subjects

In [11]:
from PIL import Image
import glob, os, tqdm
import numpy as np

data_path = '/data/mint/DPM_Dataset/HDRI_Dataset/Testset/subject_512x512/'
env = ['autumn_field_4k.exr', 'kiara_5_noon_4k.exr', 'lonely_road_afternoon_4k.exr']
sub_f = 'nf=60_maxl=2_raw'

link_folder = '/data/mint/DPM_Dataset/HDRI_Dataset/Testset_pairs/'
os.makedirs(link_folder, exist_ok=True)

for f in tqdm.tqdm(glob.glob(data_path + '/*.png')):
    src_id = f.split('/')[-1]
    src_name = src_id.split('.')[0]
    src_ext = src_id.split('.')[1]

    for e in env:
        for axis in ['azimuth', 'elevation', 'roll']:
            for fid in range(60):
                env_path = f"/Testset/{e}/{sub_f}/{axis}/hdr/{axis}_{fid:04d}.exr"
                if not os.path.exists(env_path):
                    print("[#] Missing: ", env_path)
                    assert False
                else:
                    src_link = env_path
                    tgt_link = f""
        assert False

  0%|          | 0/20 [00:00<?, ?it/s]

[#] Missing:  /Testset/autumn_field_4k.exr/nf=60_maxl=2_raw/azimuth/hdr/azimuth_0000.exr


AssertionError: 